In [1]:
%load_ext autoreload
%autoreload 2
import torch
import sys
sys.path.append('../')
from neural_control.controllers import FullyConnectedRegressionController, imitatation_learn
from neural_control.dynamics import DualSourcingModel

from dual_sourcing.utilities import sample_trajectories

In [2]:
sourcing_parameters = dict(ce=20,
                            cr=0,
                            le=0,
                            lr=2,
                            #zr=100,
                            h=5,
                            b=495,
                            T=50,
)




In [3]:
state_trajectories, qr_trajectories, qe_trajectories =\
    sample_trajectories(n_trajectories=100, optimization_samples=100, **sourcing_parameters)

costs (mean/std) [49.388 41.332 34.537 29.459 25.69  23.588 23.557] [2.9282766331759094, 2.5620730242946204, 2.1998464133716795, 1.7473844812326007, 1.861980047414498, 1.6433692908967923, 2.0927607304103053]
Delta* 6
z_e* 4


In [11]:
fcc = FullyConnectedRegressionController(lr=sourcing_parameters['lr'], 
                                         le=sourcing_parameters['le'], 
                                         n_hidden_units=[64, 64, 64, 64, 64, 64, 64, 64, 64, 64]
                                        )


In [106]:
dsd = DualSourcingModel(fcc, I_0=10, **sourcing_parameters)


In [12]:
optimizer = torch.optim.Adam(list(fcc.parameters()), lr=0.01)
losses = imitatation_learn(
    fcc,
                      lr=sourcing_parameters['lr'],
                      le=sourcing_parameters['le'],
                      optimizer=optimizer,
                      state_trajectories=state_trajectories,
                      qr_trajectories=qr_trajectories,
                      qe_trajectories=qe_trajectories,
                      minibatch_size=256,
                      epochs=1000
)

In [13]:
import plotly.express as px
px.line(losses)

In [16]:
optimizer = torch.optim.RMSprop([dsd.I_0] + list(fcc.parameters()), lr=0.0001)

In [ ]:
fine_tuning_iterations = 1000

for it in range(fine_tuning_iterations):

    dsd.reset(256, seed=1)
    total_costs = 0
    for i in range(dsd.T):
        current_costs, demands, current_inventories, qr, qra, qe, qea = dsd.simulate()
        total_costs = current_costs.mean() + total_costs
    total_costs.backward()
    optimizer.step()
    if it % 10 == 0:
        print(total_costs/dsd.T)

In [449]:
fine_tuning_iterations = 1

for it in range(fine_tuning_iterations):

    dsd.reset(2048, seed=50)
    total_costs = 0
    for i in range(dsd.T+250):
        current_costs, demands, current_inventories, qr, qra, qe, qea = dsd.simulate()
        total_costs = current_costs.mean() + total_costs
    if it % 10 == 0:
        print(total_costs/(dsd.T+250))

tensor(19.2897, grad_fn=<DivBackward0>)


In [ ]:
state_trajectories, qr_trajectories, qe_trajectories =\
    sample_trajectories(n_trajectories=100, optimization_samples=100, seed=25, **sourcing_parameters)

In [453]:
example_id = -1

In [454]:
dsd.reset(1)
fixed_demands = state_trajectories[example_id, 1:, 1].unsqueeze(-1)

nn_inv = dsd.learned_I_0.detach().clone()
dsd.reset(1)
nn_qr = [torch.zeros([1,1])]*dsd.lr
nn_qe = [torch.zeros([1,1])]*dsd.le
nn_ci = []
all_nn_inv = [nn_inv.item()]
for i in range(dsd.T):
    D = fixed_demands[i].unsqueeze(0)
    qr, qe = fcc(D, nn_inv, nn_qr, nn_qe)
    nn_qr.append(qr)
    nn_qe.append(qe)
    qra = nn_qr[-dsd.lr]
    qea = nn_qe[-1]
    c_i, nn_inv = dsd.replay_step(nn_inv, D, qra, qea, qr, qe)
    nn_ci.append(c_i)
    all_nn_inv.append(nn_inv.item())

In [455]:
dsd.reset(1)
fixed_demands = state_trajectories[example_id, 1:, 1].unsqueeze(-1)

ds_inv = state_trajectories[example_id, 0, 0].unsqueeze(0).unsqueeze(0)
dsd.reset(1)
all_qr = [0]*dsd.lr
all_qe = [0]*dsd.le
all_ci = []
all_inv = [ds_inv.detach().item()]
for i in range(dsd.T):
    D = fixed_demands[i].unsqueeze(0)
    qr = qr_trajectories[example_id, dsd.lr+i]
    qe = qe_trajectories[example_id, dsd.le+i+1]
    all_qr.append(qr)
    all_qe.append(qe)
    qra = all_qr[-dsd.lr]
    qea = all_qe[-1]
    c_i, ds_inv = dsd.replay_step(ds_inv, D, qra, qea, qr, qe)
    all_inv.append(ds_inv.detach().item())
    all_ci.append(c_i)

In [456]:
torch.tensor(all_inv) == state_trajectories[example_id, :, 0]

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True])

In [457]:
torch.stack(all_ci).mean()

tensor(26.1000)

In [458]:
torch.stack(nn_ci).mean()

tensor(15.7000, grad_fn=<MeanBackward0>)

In [459]:
torch.stack(nn_ci).mean()/torch.stack(all_ci).mean()

tensor(0.6015, grad_fn=<DivBackward0>)

In [460]:
state_trajectories[example_id, 1:, -1].mean()

tensor(26.1000)

In [461]:
 from plotly import graph_objects as go

In [462]:
a = go.Scatter(y=[x.item() for x in nn_qr], name='NNC')
b = go.Scatter(y=all_qr, name='Dual IndeX')
fig = go.Figure([a,b])
fig.layout.xaxis.title = 'Timestep'
fig.layout.yaxis.title = 'Regular Order'
fig

In [463]:
a = go.Scatter(y=[x.item() for x in nn_qe], name='NNC')
b = go.Scatter(y=all_qe, name='Dual Index')
fig = go.Figure([a,b])
fig.layout.xaxis.title = 'Timestep'
fig.layout.yaxis.title = 'Expedited Order'
fig

In [464]:
a = go.Scatter(y=[x.item() for x in nn_ci], name='NNC')
b = go.Scatter(y=[x.item() for x in all_ci], name='Dual Index')
fig = go.Figure([a,b])
fig.layout.xaxis.title = 'Timestep'
fig.layout.yaxis.title = 'Cost'
fig

In [465]:
a = go.Scatter(y=all_nn_inv, name='NNC Inventory')
b = go.Scatter(y=list(all_inv), name='Dual Index Inventory')
c = go.Scatter(y=[x.detach().item() for x in fixed_demands], name='Demand',
               x=torch.arange(1, 51).tolist() )
fig = go.Figure([a,b, c])
fig.layout.xaxis.title = 'Timestep'
fig.layout.yaxis.title = 'Quantity'
fig

costs (mean/std) [49.041 41.279 34.263 28.949 25.246 23.508 24.056] [2.978647750950492, 2.9369569560373474, 2.237226592967916, 1.8445932457927525, 1.567241673975799, 1.754456374982624, 2.134484537874248]
Delta* 5
z_e* 4
